In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 0: HARDWARE + SOFTWARE AUDIT
# REAL — KAGGLE 2×T4
# ═══════════════════════════════════════════════════════════════════
try:
    open('/kaggle/working/probe_cell0.txt', 'w').write('CELL 0 STARTED')

    import subprocess, sys, os, importlib, time

    print('=' * 70)
    print('REAL — KAGGLE 2×T4  |  CELL 0: HARDWARE + SOFTWARE AUDIT')
    print('=' * 70)

    # ── nvidia-smi ──────────────────────────────────────────────────
    print('\n[nvidia-smi]')
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout or r.stderr)

    # ── torch GPU info ──────────────────────────────────────────────
    import torch
    N_GPU = torch.cuda.device_count()
    print(f'torch.cuda.device_count() = {N_GPU}')

    GPU_INFO = []
    for i in range(N_GPU):
        p = torch.cuda.get_device_properties(i)
        vram_mb = p.total_memory // (1024 * 1024)
        bf16 = p.major >= 8
        fp16 = p.major >= 7
        fp8  = (p.major >= 8 and p.minor >= 9)
        sm_str = f'sm_{p.major}{p.minor}'
        info = {
            'idx': i,
            'name': p.name,
            'vram_mb': vram_mb,
            'major': p.major,
            'minor': p.minor,
            'sm': sm_str,
            'compute_capability': f'{p.major}.{p.minor}',
            'bf16_tensor_cores': bf16,
            'fp16_tensor_cores': fp16,
            'fp8_hw': fp8,
        }
        GPU_INFO.append(info)
        print(f'  GPU {i}: {p.name}  VRAM={vram_mb} MB  {sm_str}')
        print(f'         bf16_tensor_cores={bf16}  fp16_tensor_cores={fp16}  fp8_hw={fp8}')

    # ── driver / CUDA / cuDNN ───────────────────────────────────────
    dr = subprocess.run(
        ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print(f'\nDriver version : {dr.stdout.strip()}')
    print(f'CUDA runtime   : {torch.version.cuda}')
    print(f'cuDNN          : {torch.backends.cudnn.version()}')
    print(f'PyTorch        : {torch.__version__}')
    print(f'Python         : {sys.version}')

    # ── System resources ────────────────────────────────────────────
    print('\n[RAM — free -h]')
    print(subprocess.run(['free', '-h'], capture_output=True, text=True).stdout)
    print('[Disk — df -h /kaggle/working]')
    print(subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True).stdout)
    print('[CPU — /proc/cpuinfo]')
    cpuinfo = open('/proc/cpuinfo').read()
    for line in cpuinfo.splitlines():
        if 'model name' in line:
            print(' ', line.split(':', 1)[1].strip())
            break
    ncpu = cpuinfo.count('processor\t:')
    print(f'  CPU count: {ncpu}')

    # ── T4 compatibility analysis ────────────────────────────────────
    print('\n' + '=' * 50)
    print('T4 COMPATIBILITY ANALYSIS')
    print('=' * 50)
    for g in GPU_INFO:
        sm_label = g['sm']
        if g['major'] == 7 and g['minor'] == 5:
            print(f'  GPU {g["idx"]} ({g["name"]}) — {sm_label} detected:')
            print('  WARNING: sm_75 has NO BF16 tensor cores — Veena BF16 will use FP32 emulation (slower but functional)')
            print('  WARNING: sm_75 has NO FP8 hardware — Qwen FP8 dequantizes to BF16/FP16 at runtime (same as A6000)')
            print('  WARNING: FlashInfer may require sm_80+ — will test in vLLM startup')
            print('  WARNING: vLLM CUDA graphs may require sm_80+ — will use --enforce-eager as fallback')
        else:
            print(f'  GPU {g["idx"]} ({g["name"]}) — {sm_label} — no T4-specific warnings')

    # ── Software audit ───────────────────────────────────────────────
    print('\n' + '=' * 50)
    print('SOFTWARE AUDIT')
    print('=' * 50)

    # Pinned versions from runtime_spec.yaml
    PINNED = {
        'torch': '2.11.0',
        'vllm': '0.24.0',
        'faster_whisper': '1.2.1',
        'ctranslate2': '4.8.0',
        'transformers': '5.12.1',
        'tokenizers': None,
        'huggingface_hub': None,
        'accelerate': None,
        'snac': '1.0.0',
        'numpy': '2.3.5',
        'fastapi': None,
        'uvicorn': None,
        'pydantic': None,
        'httpx': None,
        'triton': None,
        'flashinfer': None,
    }

    sw_status = {}
    for pkg, pinned_ver in PINNED.items():
        if pkg == 'vllm':
            # NEVER import vllm in kernel — subprocess only
            r2 = subprocess.run(
                [sys.executable, '-c', 'import vllm; print(vllm.__version__)'],
                capture_output=True, text=True, timeout=30
            )
            if r2.returncode == 0:
                ver = r2.stdout.strip()
                match = (ver == pinned_ver) if pinned_ver else True
                status = 'OK' if match else f'VERSION MISMATCH (got {ver}, want {pinned_ver})'
                sw_status[pkg] = {'version': ver, 'status': status}
            else:
                sw_status[pkg] = {'version': 'NOT INSTALLED', 'status': 'MISSING'}
            print(f'  {pkg:<20} {sw_status[pkg]["version"]:<15}  [{sw_status[pkg]["status"]}]  (subprocess check)')
            continue
        try:
            mod = importlib.import_module(pkg)
            ver = getattr(mod, '__version__', 'unknown')
            if pinned_ver:
                match = (ver == pinned_ver)
                status = 'OK' if match else f'VERSION MISMATCH (got {ver}, want {pinned_ver})'
            else:
                status = 'PRESENT'
            sw_status[pkg] = {'version': ver, 'status': status}
        except ImportError:
            sw_status[pkg] = {'version': 'NOT INSTALLED', 'status': 'MISSING'}
        print(f'  {pkg:<20} {sw_status[pkg]["version"]:<15}  [{sw_status[pkg]["status"]}]')

    # ── Save hardware report ─────────────────────────────────────────
    report_lines = [
        'REAL — KAGGLE 2×T4  HARDWARE REPORT',
        f'Generated: {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}',
        '',
    ]
    for g in GPU_INFO:
        report_lines.append(f'GPU {g["idx"]}: {g["name"]}  VRAM={g["vram_mb"]} MB  {g["sm"]}  compute={g["compute_capability"]}')
        report_lines.append(f'  bf16_tensor_cores={g["bf16_tensor_cores"]}  fp16={g["fp16_tensor_cores"]}  fp8_hw={g["fp8_hw"]}')
    report_lines += [
        f'N_GPU={N_GPU}',
        f'CUDA runtime={torch.version.cuda}',
        f'cuDNN={torch.backends.cudnn.version()}',
        f'PyTorch={torch.__version__}',
        f'Python={sys.version}',
        '',
        'Software:',
    ]
    for pkg, info in sw_status.items():
        report_lines.append(f'  {pkg}: {info["version"]}  [{info["status"]}]')
    with open('/kaggle/working/hardware_report.txt', 'w') as f:
        f.write('\n'.join(report_lines) + '\n')
    print('\nHardware report saved to /kaggle/working/hardware_report.txt')

    # Store globals for later cells
    import builtins
    builtins.N_GPU = N_GPU
    builtins.GPU_INFO = GPU_INFO

    open('/kaggle/working/probe_cell0.txt', 'w').write('CELL 0 COMPLETE')
    print('\n[CELL 0 COMPLETE]')

except Exception as _e0:
    import traceback
    open('/kaggle/working/probe_cell0.txt', 'w').write(f'CELL 0 ERROR: {_e0}')
    print(f'CELL 0 ERROR: {_e0}')
    traceback.print_exc()
    raise


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1: INSTALL + SETUP
# REAL — KAGGLE 2×T4
# ═══════════════════════════════════════════════════════════════════
try:
    open('/kaggle/working/probe_cell1.txt', 'w').write('CELL 1 STARTED')

    import subprocess, sys, os, time

    print('=' * 70)
    print('REAL — KAGGLE 2×T4  |  CELL 1: INSTALL + SETUP')
    print('=' * 70)

    def _pip(pkg_spec, timeout=600):
        """Install a package quietly; return True on success."""
        print(f'  pip install {pkg_spec} ...')
        r = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', pkg_spec],
            capture_output=True, text=True, timeout=timeout
        )
        if r.returncode == 0:
            print(f'    OK')
            return True
        print(f'    FAILED: {(r.stdout + r.stderr)[-300:]}')
        return False

    # ── Pinned packages ──────────────────────────────────────────────
    # NOTE: Do NOT reinstall torch — Kaggle pre-installs it
    print('\n[Installing pinned packages — NOT touching torch]')
    for pkg in [
        'faster-whisper==1.2.1',
        'ctranslate2==4.8.0',
        'transformers==5.12.1',
        'tokenizers==0.22.2',
        'huggingface_hub==0.28.0',
        'accelerate==1.7.0',
        'snac==1.0.0',  # checkpoint trained with snac==1.0.0 architecture
        'numpy==2.3.5',
        'fastapi==0.115.6',
        'uvicorn[standard]==0.30.6',
        'pydantic==2.10.4',
        'httpx==0.27.0',
    ]:
        _pip(pkg)

    # force-reinstall snac==1.0.0 to override any Docker layer cache
    print('  Force-reinstalling snac==1.0.0 ...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall', 'snac==1.0.0'],
        capture_output=True, text=True, timeout=120
    )
    print('  snac==1.0.0 force-reinstall done')

    # ── vLLM with fallback to latest ─────────────────────────────────
    print('\n[Installing vLLM — T4 note: sm_75 needs --enforce-eager]')
    ok = _pip('vllm==0.24.0', timeout=900)
    if not ok:
        print('  vllm==0.24.0 failed — trying latest vllm ...')
        ok = _pip('vllm', timeout=900)
        if not ok:
            print('  WARNING: vllm could not be installed — LLM cell will be BLOCKED')

    # Verify vLLM via subprocess (NEVER import in kernel — CUDA segfault)
    r = subprocess.run(
        [sys.executable, '-c', 'import vllm; print("vLLM", vllm.__version__)'],
        capture_output=True, text=True, timeout=30
    )
    if r.returncode == 0:
        print(f'  subprocess vLLM check: {r.stdout.strip()}')
    else:
        print(f'  subprocess vLLM check FAILED: {r.stderr[-200:]}')

    # ── Clone VoiceOS repo ───────────────────────────────────────────
    REPO    = 'https://github.com/pateekdas7/VoiceOS.git'
    BRANCH  = 'claude/ssh-gpu-cpu-servers-y99fib'
    VOICEOS = '/kaggle/working/voiceos'

    print(f'\n[Cloning VoiceOS branch {BRANCH}]')
    if os.path.exists(VOICEOS):
        print(f'  {VOICEOS} already exists — skipping clone')
    else:
        rc = subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, VOICEOS],
            capture_output=True, text=True, timeout=180
        )
        if rc.returncode != 0:
            print(f'  CLONE FAILED: {rc.stderr[-500:]}')
        else:
            print('  Clone OK')

    print('  Verifying required files:')
    for rel in [
        'deployment/gpu/services/tts/server.py',
        'deployment/gpu/services/stt/server.py',
        'deployment/gpu/audio/pacer.py',
    ]:
        exists = os.path.exists(os.path.join(VOICEOS, rel))
        print(f'    {rel}: {"OK" if exists else "MISSING"}')

    # ── HF_HOME ─────────────────────────────────────────────────────
    HF_HOME = '/kaggle/working/hf'
    os.makedirs(HF_HOME, exist_ok=True)
    os.environ['HF_HOME'] = HF_HOME
    print(f'\nHF_HOME = {HF_HOME}')

    import builtins
    builtins.VOICEOS = VOICEOS
    builtins.HF_HOME = HF_HOME

    open('/kaggle/working/probe_cell1.txt', 'w').write('CELL 1 COMPLETE')
    print('\n[CELL 1 COMPLETE]')

except Exception as _e1:
    import traceback
    open('/kaggle/working/probe_cell1.txt', 'w').write(f'CELL 1 ERROR: {_e1}')
    print(f'CELL 1 ERROR: {_e1}')
    traceback.print_exc()
    raise


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2: STEP 4 — REAL STT VALIDATION
# REAL — KAGGLE 2×T4
# ═══════════════════════════════════════════════════════════════════
try:
    open('/kaggle/working/probe_cell2.txt', 'w').write('CELL 2 STARTED')

    import os, time, math, builtins
    import numpy as np
    import torch

    print('=' * 70)
    print('REAL — KAGGLE 2×T4  |  CELL 2: STEP 4 — REAL STT VALIDATION')
    print('=' * 70)

    N_GPU = getattr(builtins, 'N_GPU', torch.cuda.device_count())

    # GPU assignment: STT on GPU 1 if dual, else GPU 0
    if N_GPU >= 2:
        GPU_STT = '1'
        gpu_idx = 1
        print(f'GPU assignment: GPU 1 (STT — N_GPU={N_GPU} dual-GPU mode)')
    else:
        GPU_STT = '0'
        gpu_idx = 0
        print(f'GPU assignment: GPU 0 (single-GPU fallback — N_GPU={N_GPU})')
    os.environ['CUDA_VISIBLE_DEVICES'] = GPU_STT

    # ── Download Whisper ─────────────────────────────────────────────
    print('\n[Downloading Whisper large-v3-turbo]')
    from faster_whisper import WhisperModel
    from faster_whisper.utils import download_model as fw_dl
    whisper_cache = '/kaggle/working/hf/whisper'
    os.makedirs(whisper_cache, exist_ok=True)
    t0 = time.time()
    wpath = fw_dl('large-v3-turbo', cache_dir=whisper_cache)
    dl_ms = (time.time() - t0) * 1000
    print(f'  Downloaded to: {wpath}  ({dl_ms/1000:.1f}s)')
    open('/kaggle/working/probe_cell2_downloaded.txt', 'w').write(
        f'wpath={wpath}  dl_ms={dl_ms:.0f}'
    )

    # Store WPATH globally for E2E cell
    WPATH = wpath
    builtins.WPATH = WPATH
    print(f'  WPATH stored globally: {WPATH}')

    # ── VRAM before load ────────────────────────────────────────────
    torch.cuda.synchronize(gpu_idx)
    vram_before = torch.cuda.memory_allocated(gpu_idx)

    # ── Load model ──────────────────────────────────────────────────
    # int8_float16 IS compatible with T4 sm_75
    print('\n[Loading WhisperModel — compute_type=int8_float16 — T4 compatible]')
    t_load = time.time()
    whisper_model = WhisperModel(
        wpath,
        device='cuda',
        compute_type='int8_float16',
        num_workers=1,
    )
    load_ms = (time.time() - t_load) * 1000
    torch.cuda.synchronize(gpu_idx)
    vram_after = torch.cuda.memory_allocated(gpu_idx)
    vram_delta_mb = (vram_after - vram_before) / (1024 * 1024)
    print(f'  Load time  : {load_ms:.0f} ms')
    print(f'  VRAM delta : {vram_delta_mb:.1f} MB  (before={vram_before//1024//1024} MB  after={vram_after//1024//1024} MB)')

    # ── Warmup: 0.5s silence ────────────────────────────────────────
    print('\n[Warmup: 0.5s silence]')
    silence = np.zeros(int(16000 * 0.5), dtype=np.float32)
    t_warm = time.time()
    segs, _ = whisper_model.transcribe(silence, language='hi')
    _ = list(segs)
    warmup_ms = (time.time() - t_warm) * 1000
    print(f'  Warmup: {warmup_ms:.0f} ms')

    # ── 3 trials: 2s sine at 16kHz ──────────────────────────────────
    print('\n[STT inference: 3 trials on 2s 440Hz sine at 16kHz (simulated speech)]')
    t_arr = np.linspace(0.0, 2.0, 32000, dtype=np.float32)
    test_audio = (np.sin(2.0 * math.pi * 440.0 * t_arr) * 0.3).astype(np.float32)
    trial_latencies = []
    for i in range(3):
        t_trial = time.time()
        segs, info = whisper_model.transcribe(test_audio, language='hi')
        _ = list(segs)
        lat = (time.time() - t_trial) * 1000
        trial_latencies.append(lat)
        print(f'  Trial {i+1}: {lat:.0f} ms  lang={info.language}  prob={info.language_probability:.3f}')

    avg_latency_ms = sum(trial_latencies) / len(trial_latencies)
    print(f'  Avg latency: {avg_latency_ms:.0f} ms')

    # ── Results ──────────────────────────────────────────────────────
    STT_RESULTS = {
        'label': 'REAL — KAGGLE 2×T4',
        'model': 'large-v3-turbo',
        'compute_type': 'int8_float16',
        'gpu': f'GPU {GPU_STT}',
        'load_ms': load_ms,
        'warmup_ms': warmup_ms,
        'trial_latencies_ms': trial_latencies,
        'avg_latency_ms': avg_latency_ms,
        'vram_delta_mb': vram_delta_mb,
    }
    STT_STATUS = 'PASS'
    builtins.STT_RESULTS = STT_RESULTS
    builtins.STT_STATUS = STT_STATUS
    print(f'\nSTT_STATUS = {STT_STATUS}')
    print(f'STT_RESULTS = {STT_RESULTS}')

    # ── Cleanup ──────────────────────────────────────────────────────
    del whisper_model
    torch.cuda.empty_cache()
    print('  Model freed')

    open('/kaggle/working/probe_cell2_done.txt', 'w').write(
        f'STT PASS  avg={avg_latency_ms:.0f}ms  vram_delta={vram_delta_mb:.1f}MB'
    )
    print('\n[CELL 2 COMPLETE — REAL — KAGGLE 2×T4]')

except Exception as _e2:
    import traceback, builtins
    STT_STATUS = f'FAIL — {_e2}'
    STT_RESULTS = {'error': str(_e2), 'label': 'REAL — KAGGLE 2×T4'}
    builtins.STT_STATUS = STT_STATUS
    builtins.STT_RESULTS = STT_RESULTS
    open('/kaggle/working/probe_cell2.txt', 'w').write(f'CELL 2 ERROR: {_e2}')
    print(f'CELL 2 ERROR: {_e2}')
    traceback.print_exc()


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3: STEP 5 — REAL LLM VALIDATION
# REAL — KAGGLE 2×T4
# ═══════════════════════════════════════════════════════════════════
_vllm_proc3 = None
try:
    open('/kaggle/working/probe_cell3.txt', 'w').write('CELL 3 STARTED')

    import subprocess, sys, os, time, json, builtins
    import torch
    import httpx

    print('=' * 70)
    print('REAL — KAGGLE 2×T4  |  CELL 3: STEP 5 — REAL LLM VALIDATION')
    print('=' * 70)
    print('NOTE: NEVER import vllm in kernel — subprocess only (avoids CUDA segfault)')

    # LLM always on GPU 0
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    print('GPU assignment: GPU 0 (LLM — always)')

    # ── Check vLLM via subprocess ────────────────────────────────────
    print('\n[Checking vLLM availability via subprocess]')
    rv = subprocess.run(
        [sys.executable, '-c', 'import vllm; print(vllm.__version__)'],
        capture_output=True, text=True, timeout=30
    )
    if rv.returncode != 0:
        LLM_STATUS = 'BLOCKED — vLLM not installable on T4 environment'
        LLM_RESULTS = {'label': 'REAL — KAGGLE 2×T4', 'status': LLM_STATUS, 'stderr': rv.stderr[-300:]}
        builtins.LLM_STATUS = LLM_STATUS
        builtins.LLM_RESULTS = LLM_RESULTS
        builtins.QPATH = None
        print(f'  {LLM_STATUS}')
        open('/kaggle/working/probe_cell3_done.txt', 'w').write(LLM_STATUS)
    else:
        vllm_ver = rv.stdout.strip()
        print(f'  vLLM {vllm_ver} available')

        # ── Download Qwen ────────────────────────────────────────────
        open('/kaggle/working/probe_cell3_download_start.txt', 'w').write(
            'Qwen2.5-7B-FP8 download started'
        )
        print('\n[Downloading RedHatAI/Qwen2.5-7B-Instruct-FP8-dynamic]')
        print('  T4 note: sm_75 no FP8 HW — vLLM dequantizes to BF16/FP16 (same as A6000)')
        from huggingface_hub import snapshot_download
        t_dl = time.time()
        qpath = snapshot_download(
            'RedHatAI/Qwen2.5-7B-Instruct-FP8-dynamic',
            cache_dir='/kaggle/working/hf',
            ignore_patterns=['*.pt', '*.gguf'],
        )
        dl_s = time.time() - t_dl
        print(f'  Downloaded to {qpath}  ({dl_s:.1f}s)')
        open('/kaggle/working/probe_cell3_downloaded.txt', 'w').write(
            f'qpath={qpath}  dl_s={dl_s:.1f}'
        )
        QPATH = qpath
        builtins.QPATH = QPATH

        # ── Start vLLM subprocess ────────────────────────────────────
        env3 = os.environ.copy()
        env3['CUDA_VISIBLE_DEVICES'] = '0'
        log3 = '/kaggle/working/vllm_standalone.log'
        log3_fh = open(log3, 'w')

        base_cmd3 = [
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--model', qpath,
            '--dtype', 'auto',
            '--port', '8000',
            '--host', '0.0.0.0',
            '--max-model-len', '512',
            '--gpu-memory-utilization', '0.85',
            '--served-model-name', 'qwen2.5-7b-instruct-fp8',
            '--trust-remote-code',
        ]
        print(f'\n[Starting vLLM — log: {log3}]')
        _vllm_proc3 = subprocess.Popen(base_cmd3, stdout=log3_fh, stderr=log3_fh, env=env3)
        t_vllm_start = time.time()
        healthy3 = False
        last_rpt3 = t_vllm_start
        retry_done3 = False

        # Poll phase 1
        while (time.time() - t_vllm_start) < 300:
            if _vllm_proc3.poll() is not None and not retry_done3:
                log3_fh.flush()
                tail3 = open(log3).read()[-2000:]
                print(f'  vLLM exited early (rc={_vllm_proc3.returncode}) — retrying with --enforce-eager --dtype float16')
                print(f'  Log tail:\n{tail3[-800:]}')
                log3_fh.close()
                log3_fh = open(log3, 'a')
                fallback_cmd3 = base_cmd3 + ['--enforce-eager', '--dtype', 'float16']
                _vllm_proc3 = subprocess.Popen(fallback_cmd3, stdout=log3_fh, stderr=log3_fh, env=env3)
                t_vllm_start = time.time()
                retry_done3 = True
                break
            if time.time() - last_rpt3 >= 30:
                print(f'  Waiting for vLLM... {time.time()-t_vllm_start:.0f}s')
                last_rpt3 = time.time()
            try:
                resp3 = httpx.get('http://localhost:8000/health', timeout=3)
                if resp3.status_code == 200:
                    startup_ms3 = (time.time() - t_vllm_start) * 1000
                    healthy3 = True
                    print(f'  vLLM healthy after {startup_ms3:.0f} ms')
                    break
            except Exception:
                pass
            time.sleep(2)

        # Poll phase 2 (after retry)
        if not healthy3:
            last_rpt3 = time.time()
            t_retry3 = time.time()
            while (time.time() - t_retry3) < 300:
                if _vllm_proc3.poll() is not None:
                    log3_fh.flush()
                    tail3 = open(log3).read()[-2000:]
                    print(f'  vLLM exited on retry (rc={_vllm_proc3.returncode})')
                    print(f'  Log tail:\n{tail3[-800:]}')
                    LLM_STATUS = 'BLOCKED — vLLM could not start on T4 sm_75: see vllm_standalone.log'
                    LLM_RESULTS = {'label': 'REAL — KAGGLE 2×T4', 'status': LLM_STATUS}
                    builtins.LLM_STATUS = LLM_STATUS
                    builtins.LLM_RESULTS = LLM_RESULTS
                    open('/kaggle/working/probe_cell3_done.txt', 'w').write(LLM_STATUS)
                    print(f'\nLLM_STATUS = {LLM_STATUS}')
                    raise SystemExit(LLM_STATUS)
                if time.time() - last_rpt3 >= 30:
                    print(f'  Waiting for vLLM (retry)... {time.time()-t_retry3:.0f}s')
                    last_rpt3 = time.time()
                try:
                    resp3 = httpx.get('http://localhost:8000/health', timeout=3)
                    if resp3.status_code == 200:
                        startup_ms3 = (time.time() - t_retry3) * 1000
                        healthy3 = True
                        print(f'  vLLM healthy (retry) after {startup_ms3:.0f} ms')
                        break
                except Exception:
                    pass
                time.sleep(2)

        if not healthy3:
            tail3 = open(log3).read()[-1000:]
            print(f'  vLLM timeout\n{tail3}')
            LLM_STATUS = 'BLOCKED — vLLM timeout on T4 sm_75'
            LLM_RESULTS = {'label': 'REAL — KAGGLE 2×T4', 'status': LLM_STATUS}
            builtins.LLM_STATUS = LLM_STATUS
            builtins.LLM_RESULTS = LLM_RESULTS
            open('/kaggle/working/probe_cell3_done.txt', 'w').write(LLM_STATUS)
            raise SystemExit(LLM_STATUS)

        # ── Query models ─────────────────────────────────────────────
        models_resp3 = httpx.get('http://localhost:8000/v1/models', timeout=10)
        print(f'  /v1/models: {models_resp3.json()}')

        # ── Streaming inference ──────────────────────────────────────
        print('\n[Streaming inference — Hindi prompt]')
        prompt3 = 'नमस्ते, एक वाक्य में जवाब दें।'
        payload3 = {
            'model': 'qwen2.5-7b-instruct-fp8',
            'messages': [{'role': 'user', 'content': prompt3}],
            'max_tokens': 50,
            'stream': True,
        }
        t_infer3 = time.time()
        first_tok3 = None
        resp_text3 = ''
        with httpx.stream(
            'POST', 'http://localhost:8000/v1/chat/completions',
            json=payload3, timeout=120
        ) as sr3:
            for line3 in sr3.iter_lines():
                if line3.startswith('data: '):
                    data3 = line3[6:]
                    if data3 == '[DONE]':
                        break
                    try:
                        ch3 = json.loads(data3)
                        delta3 = ch3['choices'][0]['delta'].get('content', '')
                        if delta3:
                            if first_tok3 is None:
                                first_tok3 = time.time()
                            resp_text3 += delta3
                    except Exception:
                        pass
        total3_ms = (time.time() - t_infer3) * 1000
        ttft3_ms = (first_tok3 - t_infer3) * 1000 if first_tok3 else None
        print(f'  TTFT:           {ttft3_ms:.0f} ms' if ttft3_ms else '  TTFT: None')
        print(f'  Total latency:  {total3_ms:.0f} ms')
        print(f'  Response:       {resp_text3!r}')

        LLM_RESULTS = {
            'label': 'REAL — KAGGLE 2×T4',
            'model': 'RedHatAI/Qwen2.5-7B-Instruct-FP8-dynamic',
            'vllm_version': vllm_ver,
            'gpu': 'GPU 0',
            'startup_ms': startup_ms3,
            'prompt': prompt3,
            'response': resp_text3,
            'ttft_ms': ttft3_ms,
            'total_latency_ms': total3_ms,
            'note': 'FP8 dequantized to BF16/FP16 on T4 sm_75 — no FP8 HW cores',
        }
        LLM_STATUS = 'PASS'
        builtins.LLM_STATUS = LLM_STATUS
        builtins.LLM_RESULTS = LLM_RESULTS
        print(f'\nLLM_STATUS = {LLM_STATUS}')
        print(f'LLM_RESULTS = {LLM_RESULTS}')
        open('/kaggle/working/probe_cell3_done.txt', 'w').write(
            f'LLM PASS  ttft={ttft3_ms:.0f}ms  total={total3_ms:.0f}ms' if ttft3_ms else 'LLM PASS'
        )
        print('\n[CELL 3 COMPLETE — REAL — KAGGLE 2×T4]')

except SystemExit as _se3:
    print(str(_se3))
except Exception as _e3:
    import traceback, builtins
    LLM_STATUS = f'FAIL — {_e3}'
    LLM_RESULTS = {'error': str(_e3), 'label': 'REAL — KAGGLE 2×T4'}
    builtins.LLM_STATUS = LLM_STATUS
    builtins.LLM_RESULTS = LLM_RESULTS
    open('/kaggle/working/probe_cell3.txt', 'w').write(f'CELL 3 ERROR: {_e3}')
    print(f'CELL 3 ERROR: {_e3}')
    traceback.print_exc()
finally:
    if _vllm_proc3 and _vllm_proc3.poll() is None:
        _vllm_proc3.terminate()
        try:
            _vllm_proc3.wait(timeout=15)
        except Exception:
            _vllm_proc3.kill()
        print('  vLLM subprocess terminated')
    import torch as _t3; _t3.cuda.empty_cache()
    print('  CUDA cache emptied')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4: STEPS 6+7 — REAL TTS + AUDIO PACER VALIDATION
# REAL — KAGGLE 2×T4
# ═══════════════════════════════════════════════════════════════════
_tts_proc4 = None
try:
    open('/kaggle/working/probe_cell4.txt', 'w').write('CELL 4 STARTED')

    import subprocess, sys, os, time, wave, struct, math, audioop, builtins
    import numpy as np
    import torch
    import httpx

    print('=' * 70)
    print('REAL — KAGGLE 2×T4  |  CELL 4: STEPS 6+7 — REAL TTS + AUDIO PACER')
    print('=' * 70)

    N_GPU = getattr(builtins, 'N_GPU', torch.cuda.device_count())
    VOICEOS = getattr(builtins, 'VOICEOS', '/kaggle/working/voiceos')

    # GPU assignment
    if N_GPU >= 2:
        GPU_TTS4 = '1'
        gpu_idx4 = 1
        print(f'GPU assignment: GPU 1 (TTS+STT — N_GPU={N_GPU})')
    else:
        GPU_TTS4 = '0'
        gpu_idx4 = 0
        print(f'GPU assignment: GPU 0 (single-GPU fallback)')

    env4 = os.environ.copy()
    env4['CUDA_VISIBLE_DEVICES'] = GPU_TTS4

    tts_server4 = os.path.join(VOICEOS, 'deployment/gpu/services/tts/server.py')
    tts_log4 = '/kaggle/working/tts_standalone.log'
    tts_fh4 = open(tts_log4, 'w')
    tts_cmd4 = [
        sys.executable, tts_server4,
        '--model-path', 'maya-research/Veena',
        '--snac-path', 'hubertsiuzdak/snac_24khz',
        '--device', 'cuda',
        '--port', '8200',
    ]
    print(f'\n[Starting TTS server — log: {tts_log4}]')
    _tts_proc4 = subprocess.Popen(tts_cmd4, stdout=tts_fh4, stderr=tts_fh4, env=env4)
    open('/kaggle/working/probe_cell4_tts_started.txt', 'w').write('TTS Popen done')
    t_tts_launch4 = time.time()

    # ── Poll /health/ready ───────────────────────────────────────────
    print('\n[Polling TTS /health/ready — timeout=700s]')
    tts_ready4 = False
    last_rpt4 = t_tts_launch4
    ready_data4 = {}

    while (time.time() - t_tts_launch4) < 700:
        if _tts_proc4.poll() is not None:
            tts_fh4.flush()
            tail4 = open(tts_log4).read()[-2000:]
            print(f'  TTS exited early (rc={_tts_proc4.returncode})')
            print(f'  Log tail:\n{tail4[-800:]}')
            TTS_STATUS = 'BLOCKED — TTS server exited before ready'
            TTS_RESULTS = {'label': 'REAL — KAGGLE 2×T4', 'status': TTS_STATUS}
            PACER_STATUS = 'BLOCKED — TTS not ready'
            PACER_RESULTS = {'label': 'REAL — KAGGLE 2×T4', 'status': PACER_STATUS}
            builtins.TTS_STATUS = TTS_STATUS; builtins.TTS_RESULTS = TTS_RESULTS
            builtins.PACER_STATUS = PACER_STATUS; builtins.PACER_RESULTS = PACER_RESULTS
            open('/kaggle/working/probe_cell4_done.txt', 'w').write(TTS_STATUS)
            raise SystemExit(TTS_STATUS)
        if time.time() - last_rpt4 >= 30:
            print(f'  Waiting for TTS... {time.time()-t_tts_launch4:.0f}s')
            last_rpt4 = time.time()
        try:
            r4 = httpx.get('http://localhost:8200/health/ready', timeout=3)
            if r4.status_code == 200:
                tts_startup_ms4 = (time.time() - t_tts_launch4) * 1000
                tts_ready4 = True
                ready_data4 = r4.json()
                print(f'  TTS ready after {tts_startup_ms4:.0f} ms')
                break
        except Exception:
            pass
        time.sleep(3)

    if not tts_ready4:
        tail4 = open(tts_log4).read()[-1000:]
        print(f'  TTS timeout\n{tail4}')
        TTS_STATUS = 'BLOCKED — TTS timeout (>700s) on T4'
        TTS_RESULTS = {'label': 'REAL — KAGGLE 2×T4', 'status': TTS_STATUS}
        PACER_STATUS = 'BLOCKED — TTS not ready'; PACER_RESULTS = {'label': 'REAL — KAGGLE 2×T4', 'status': PACER_STATUS}
        builtins.TTS_STATUS = TTS_STATUS; builtins.TTS_RESULTS = TTS_RESULTS
        builtins.PACER_STATUS = PACER_STATUS; builtins.PACER_RESULTS = PACER_RESULTS
        open('/kaggle/working/probe_cell4_done.txt', 'w').write(TTS_STATUS)
        raise SystemExit(TTS_STATUS)

    # ── Validate health/ready response ───────────────────────────────
    print(f'  /health/ready: {ready_data4}')
    assert ready_data4.get('encoding') == 'pcm16le', \
        f'Expected encoding=pcm16le, got {ready_data4.get("encoding")}'
    assert ready_data4.get('chunk_bytes') == 4096, \
        f'Expected chunk_bytes=4096, got {ready_data4.get("chunk_bytes")}'
    if ready_data4.get('mock') is True:
        TTS_STATUS = 'BLOCKED — TTS running in mock mode'
        TTS_RESULTS = {'label': 'REAL — KAGGLE 2×T4', 'status': TTS_STATUS, 'health': ready_data4}
        PACER_STATUS = 'BLOCKED — TTS mock'
        PACER_RESULTS = {'label': 'REAL — KAGGLE 2×T4', 'status': PACER_STATUS}
        builtins.TTS_STATUS = TTS_STATUS; builtins.TTS_RESULTS = TTS_RESULTS
        builtins.PACER_STATUS = PACER_STATUS; builtins.PACER_RESULTS = PACER_RESULTS
        open('/kaggle/working/probe_cell4_done.txt', 'w').write(TTS_STATUS)
        raise SystemExit(TTS_STATUS)
    print(f'  Encoding OK: pcm16le  chunk_bytes=4096  mock={ready_data4.get("mock", False)}')

    # ── STEP 6: Synthesize mandatory sentence ────────────────────────
    MANDATORY_TEXT4 = 'नमस्ते सर, मैं Kavya बोल रही हूं। आपके loan account के बारे में बात करनी थी। क्या आप थोड़ा वक्त दे सकते हैं?'
    print(f'\n[STEP 6: Synthesize mandatory sentence]')
    print(f'  Text: {MANDATORY_TEXT4}')

    chunks4 = []
    chunk_sizes4 = []
    t_synth4 = time.time()
    first_chunk4_time = None

    with httpx.stream(
        'POST', 'http://localhost:8200/synthesize',
        json={'text': MANDATORY_TEXT4, 'speaker': 'Kavya'},
        timeout=180
    ) as sr4:
        for chunk4 in sr4.iter_bytes(chunk_size=4096):
            if chunk4:
                if first_chunk4_time is None:
                    first_chunk4_time = time.time()
                chunks4.append(chunk4)
                chunk_sizes4.append(len(chunk4))

    ttfa4_ms = (first_chunk4_time - t_synth4) * 1000 if first_chunk4_time else None
    total_synth4_ms = (time.time() - t_synth4) * 1000
    total_bytes4 = sum(len(c) for c in chunks4)
    print(f'  TTFA:         {ttfa4_ms:.0f} ms' if ttfa4_ms else '  TTFA: N/A')
    print(f'  Total synth:  {total_synth4_ms:.0f} ms')
    print(f'  Chunks:       {len(chunks4)}')
    print(f'  Total bytes:  {total_bytes4}')
    print(f'  Chunk sizes:  {sorted(set(chunk_sizes4))}')

    # ── STEP 7: Float32 bug verification ─────────────────────────────
    print('\n[STEP 7: "Na--mas--te" float32 bug verification]')
    bugs4 = []
    for i4, chunk4 in enumerate(chunks4):
        if len(chunk4) % 2 != 0:
            bugs4.append(f'Chunk {i4}: NOT 16-bit aligned (len={len(chunk4)})')
        if len(chunk4) == 8192:
            bugs4.append(f'Chunk {i4}: len=8192 — FLOAT32 BUG PRESENT')
        n4 = len(chunk4) // 2
        samples4 = struct.unpack(f'<{n4}h', chunk4[:n4*2])
        oob4 = [s for s in samples4 if not (-32768 <= s <= 32767)]
        if oob4:
            bugs4.append(f'Chunk {i4}: samples out of int16 range: {oob4[:3]}')

    non_std4 = [s for s in chunk_sizes4 if s != 4096 and s > 512]
    if non_std4:
        print(f'  WARNING: non-standard chunk sizes: {non_std4}')
    else:
        print(f'  All chunks 4096 bytes (PCM16LE contract) or small partials')

    if bugs4:
        for b4 in bugs4:
            print(f'  BUG: {b4}')
        float32_bug4 = True
    else:
        float32_bug4 = False
        print('  No float32 bug — all chunks 16-bit aligned, no 8192-byte chunks')
        print('  All PCM16LE samples in valid int16 range')

    audio_dur4_s = total_bytes4 / (24000 * 2)
    print(f'  Audio duration: {audio_dur4_s*1000:.0f} ms ({audio_dur4_s:.3f} s)')
    print(f'  Verification: 4096 bytes / 2 = 2048 int16 samples = {2048/24000*1000:.2f}ms per chunk')

    # VRAM on TTS GPU
    torch.cuda.synchronize(gpu_idx4)
    vram_tts4_mb = torch.cuda.memory_allocated(gpu_idx4) / (1024 * 1024)
    print(f'  VRAM GPU {GPU_TTS4}: {vram_tts4_mb:.1f} MB allocated')

    # ── AudioPacer validation ────────────────────────────────────────
    print('\n[AudioPacer validation]')
    sys.path.insert(0, VOICEOS)
    from deployment.gpu.audio.pacer import AudioPacer

    pacer4 = AudioPacer()
    for c4 in chunks4:
        pacer4.feed(c4)

    frames4 = list(pacer4.drain_all_frames())
    for f4 in frames4:
        assert len(f4) == 160, f'Frame len {len(f4)} != 160'

    # Underrun count (attribute may vary by version)
    underruns4 = getattr(pacer4, 'underrun_count', 0)
    silence_frames4 = getattr(pacer4, 'frames_silence', 0)
    total_frames4 = len(frames4)
    total_audio4_ms = total_frames4 * 20

    print(f'  Total frames: {total_frames4}')
    print(f'  Total audio : {total_audio4_ms} ms ({total_audio4_ms/1000:.3f} s)')
    print(f'  Underruns   : {underruns4}')
    print(f'  Silence frm : {silence_frames4}')
    print(f'  All frames 160 bytes: OK')
    print(f'  Verification: 4096/2=2048 int16 samples={2048/24000*1000:.2f}ms/chunk')
    print(f'  Verification: 160 bytes μ-law at 8kHz = {160/8000*1000:.0f}ms per frame')

    # ── Cancel test ──────────────────────────────────────────────────
    print('\n[Cancel test]')
    pacer4b = AudioPacer()
    for c4b in chunks4[:3]:
        pacer4b.feed(c4b)
    pacer4b.cancel()
    result4b = pacer4b.drain_frame()
    assert result4b is None, f'Expected None after cancel(), got {result4b!r}'
    print('  Cancel test PASS — drain_frame() returns None after cancel()')

    # ── Save artifacts ───────────────────────────────────────────────
    print('\n[Saving audio artifacts]')
    all_pcm4 = b''.join(chunks4)

    with open('/kaggle/working/veena_output_24khz_pcm16le.raw', 'wb') as f:
        f.write(all_pcm4)
    print(f'  Saved: veena_output_24khz_pcm16le.raw ({len(all_pcm4)} bytes)')

    with wave.open('/kaggle/working/veena_output_24khz.wav', 'wb') as w4:
        w4.setnchannels(1)
        w4.setsampwidth(2)
        w4.setframerate(24000)
        w4.writeframes(all_pcm4)
    print('  Saved: veena_output_24khz.wav')

    all_ulaw4 = b''.join(frames4)
    with open('/kaggle/working/veena_output_8khz.ulaw', 'wb') as f:
        f.write(all_ulaw4)
    print(f'  Saved: veena_output_8khz.ulaw ({len(all_ulaw4)} bytes)')

    # ── Results ──────────────────────────────────────────────────────
    TTS_RESULTS = {
        'label': 'REAL — KAGGLE 2×T4',
        'model': 'maya-research/Veena',
        'gpu': f'GPU {GPU_TTS4}',
        'startup_ms': tts_startup_ms4,
        'ttfa_ms': ttfa4_ms,
        'total_synth_ms': total_synth4_ms,
        'chunks': len(chunks4),
        'total_bytes': total_bytes4,
        'audio_duration_ms': audio_dur4_s * 1000,
        'float32_bug': float32_bug4,
        'chunk_sizes_ok': len(non_std4) == 0,
        'vram_mb': vram_tts4_mb,
        'health': ready_data4,
    }
    TTS_STATUS = 'FAIL — float32 bug detected' if float32_bug4 else 'PASS'

    PACER_RESULTS = {
        'label': 'REAL — KAGGLE 2×T4',
        'total_frames': total_frames4,
        'total_audio_ms': total_audio4_ms,
        'underruns': underruns4,
        'silence_frames': silence_frames4,
        'cancel_test': 'PASS',
    }
    PACER_STATUS = 'PASS' if underruns4 == 0 else f'FAIL — {underruns4} underruns'

    builtins.TTS_RESULTS = TTS_RESULTS
    builtins.TTS_STATUS = TTS_STATUS
    builtins.PACER_RESULTS = PACER_RESULTS
    builtins.PACER_STATUS = PACER_STATUS
    builtins.ALL_CHUNKS4 = chunks4
    builtins.ALL_FRAMES4 = frames4

    print(f'\nTTS_STATUS   = {TTS_STATUS}')
    print(f'PACER_STATUS = {PACER_STATUS}')
    open('/kaggle/working/probe_cell4_done.txt', 'w').write(
        f'TTS={TTS_STATUS}  PACER={PACER_STATUS}'
    )
    print('\n[CELL 4 COMPLETE — REAL — KAGGLE 2×T4]')

except SystemExit as _se4:
    print(str(_se4))
except Exception as _e4:
    import traceback, builtins
    TTS_STATUS = f'FAIL — {_e4}'
    TTS_RESULTS = {'error': str(_e4), 'label': 'REAL — KAGGLE 2×T4'}
    PACER_STATUS = f'FAIL — {_e4}'
    PACER_RESULTS = {'error': str(_e4), 'label': 'REAL — KAGGLE 2×T4'}
    builtins.TTS_STATUS = TTS_STATUS; builtins.TTS_RESULTS = TTS_RESULTS
    builtins.PACER_STATUS = PACER_STATUS; builtins.PACER_RESULTS = PACER_RESULTS
    open('/kaggle/working/probe_cell4.txt', 'w').write(f'CELL 4 ERROR: {_e4}')
    print(f'CELL 4 ERROR: {_e4}')
    traceback.print_exc()
finally:
    if _tts_proc4 and _tts_proc4.poll() is None:
        _tts_proc4.terminate()
        try:
            _tts_proc4.wait(timeout=15)
        except Exception:
            _tts_proc4.kill()
        print('  TTS subprocess terminated')
    import torch as _t4; _t4.cuda.empty_cache()
    print('  CUDA cache emptied')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5: STEPS 8+9 — REAL E2E PIPELINE
# REAL — KAGGLE 2×T4
# ═══════════════════════════════════════════════════════════════════
_e2e_llm5 = None
_e2e_stt5 = None
_e2e_tts5 = None
try:
    open('/kaggle/working/probe_cell5.txt', 'w').write('CELL 5 STARTED')

    import subprocess, sys, os, time, json, math, builtins
    import numpy as np
    import torch
    import httpx

    print('=' * 70)
    print('REAL — KAGGLE 2×T4  |  CELL 5: STEPS 8+9 — REAL E2E PIPELINE')
    print('=' * 70)

    N_GPU = getattr(builtins, 'N_GPU', torch.cuda.device_count())
    VOICEOS = getattr(builtins, 'VOICEOS', '/kaggle/working/voiceos')
    WPATH = getattr(builtins, 'WPATH', 'large-v3-turbo')
    QPATH = getattr(builtins, 'QPATH', None)

    if N_GPU >= 2:
        GPU_LLM5   = '0'
        GPU_STT_TTS5 = '1'
        MODE5 = 'PARALLEL dual-GPU'
    else:
        GPU_LLM5   = '0'
        GPU_STT_TTS5 = '0'
        MODE5 = 'SEQUENTIAL single-GPU fallback'

    print(f'GPU layout: {MODE5}')
    print(f'  LLM     → GPU {GPU_LLM5}')
    print(f'  STT+TTS → GPU {GPU_STT_TTS5}')

    env_llm5 = os.environ.copy(); env_llm5['CUDA_VISIBLE_DEVICES'] = GPU_LLM5
    env_sst5 = os.environ.copy(); env_sst5['CUDA_VISIBLE_DEVICES'] = GPU_STT_TTS5

    # ── Start LLM ────────────────────────────────────────────────────
    llm_log5 = '/kaggle/working/e2e_llm.log'
    llm_fh5 = open(llm_log5, 'w')
    if QPATH:
        llm_cmd5 = [
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--model', QPATH, '--dtype', 'auto',
            '--port', '8000', '--host', '0.0.0.0',
            '--max-model-len', '512', '--gpu-memory-utilization', '0.85',
            '--served-model-name', 'qwen2.5-7b-instruct-fp8',
            '--trust-remote-code', '--enforce-eager',
        ]
        print(f'\n[Starting LLM — GPU {GPU_LLM5} — {llm_log5}]')
        _e2e_llm5 = subprocess.Popen(llm_cmd5, stdout=llm_fh5, stderr=llm_fh5, env=env_llm5)
        print(f'  LLM pid={_e2e_llm5.pid}')
    else:
        print('  LLM SKIPPED — QPATH not set (Cell 3 was BLOCKED)')

    # ── Start STT ────────────────────────────────────────────────────
    stt_log5 = '/kaggle/working/e2e_stt.log'
    stt_fh5 = open(stt_log5, 'w')
    stt_srv5 = os.path.join(VOICEOS, 'deployment/gpu/services/stt/server.py')
    stt_cmd5 = [
        sys.executable, stt_srv5,
        '--model-path', WPATH,
        '--compute-type', 'int8_float16',
        '--device', 'cuda', '--port', '8100',
    ]
    print(f'\n[Starting STT — GPU {GPU_STT_TTS5} — {stt_log5}]')
    _e2e_stt5 = subprocess.Popen(stt_cmd5, stdout=stt_fh5, stderr=stt_fh5, env=env_sst5)
    print(f'  STT pid={_e2e_stt5.pid}')

    # ── Start TTS ────────────────────────────────────────────────────
    tts_log5 = '/kaggle/working/e2e_tts.log'
    tts_fh5 = open(tts_log5, 'w')
    tts_srv5 = os.path.join(VOICEOS, 'deployment/gpu/services/tts/server.py')
    tts_cmd5 = [
        sys.executable, tts_srv5,
        '--model-path', 'maya-research/Veena',
        '--snac-path', 'hubertsiuzdak/snac_24khz',
        '--device', 'cuda', '--port', '8200',
    ]
    print(f'\n[Starting TTS — GPU {GPU_STT_TTS5} — {tts_log5}]')
    _e2e_tts5 = subprocess.Popen(tts_cmd5, stdout=tts_fh5, stderr=tts_fh5, env=env_sst5)
    print(f'  TTS pid={_e2e_tts5.pid}')

    # ── Wait for all services ────────────────────────────────────────
    def _wait5(url, timeout_s, label):
        t5 = time.time()
        last5 = t5
        while (time.time() - t5) < timeout_s:
            try:
                r5 = httpx.get(url, timeout=3)
                if r5.status_code == 200:
                    ms5 = (time.time() - t5) * 1000
                    print(f'  {label}: ready in {ms5:.0f} ms')
                    return True, ms5
            except Exception:
                pass
            if time.time() - last5 >= 30:
                print(f'  {label}: waiting... {time.time()-t5:.0f}s')
                last5 = time.time()
            time.sleep(2)
        print(f'  {label}: TIMEOUT after {timeout_s}s')
        return False, timeout_s * 1000

    print('\n[Waiting for all services]')
    stt_ok5, stt_ms5 = _wait5('http://localhost:8100/health/ready', 120, f'STT GPU{GPU_STT_TTS5}')
    tts_ok5, tts_ms5 = _wait5('http://localhost:8200/health/ready', 700, f'TTS GPU{GPU_STT_TTS5}')
    if _e2e_llm5:
        llm_ok5, llm_ms5 = _wait5('http://localhost:8000/health', 300, f'LLM GPU{GPU_LLM5}')
    else:
        llm_ok5, llm_ms5 = False, 0

    print(f'\nService status: LLM={llm_ok5}  STT={stt_ok5}  TTS={tts_ok5}')
    print(f'  LLM GPU {GPU_LLM5}    → {"READY" if llm_ok5 else "NOT READY"}')
    print(f'  STT GPU {GPU_STT_TTS5}    → {"READY" if stt_ok5 else "NOT READY"}')
    print(f'  TTS GPU {GPU_STT_TTS5}    → {"READY" if tts_ok5 else "NOT READY"}')

    # VRAM before conversation
    print('\n[VRAM after services loaded]')
    if N_GPU >= 2:
        for gi5 in range(N_GPU):
            v5 = torch.cuda.memory_allocated(gi5) / (1024 * 1024)
            print(f'  GPU {gi5} VRAM: {v5:.1f} MB')
    else:
        v5 = torch.cuda.memory_allocated(0) / (1024 * 1024)
        print(f'  GPU 0 VRAM: {v5:.1f} MB')

    # ── Run real conversation ────────────────────────────────────────
    print('\n[E2E conversation: 2s 440Hz sine → STT → LLM → TTS → AudioPacer]')
    t_pipe5 = time.time()

    # Input audio: 2s 440Hz sine at 16kHz
    t5arr = np.linspace(0.0, 2.0, 32000, dtype=np.float32)
    audio5 = (np.sin(2.0 * math.pi * 440.0 * t5arr) * 0.3).astype(np.float32)
    pcm16_5 = (audio5 * 32767).astype(np.int16).tobytes()
    import base64
    audio5_b64 = base64.b64encode(pcm16_5).decode()

    # STT
    stt_text5 = ''
    stt_lat5 = 0.0
    if stt_ok5:
        t5s = time.time()
        r5s = httpx.post(
            'http://localhost:8100/transcribe',
            json={'audio_b64': audio5_b64, 'sample_rate': 16000, 'language': 'hi'},
            timeout=30
        )
        stt_lat5 = (time.time() - t5s) * 1000
        if r5s.status_code == 200:
            stt_text5 = r5s.json().get('text', '')
        print(f'  STT: {stt_lat5:.0f}ms → {stt_text5!r}')
    else:
        stt_text5 = 'नमस्ते'
        print(f'  STT: BLOCKED — fallback text: {stt_text5!r}')

    # LLM
    llm_resp5 = ''
    llm_ttft5 = 0.0
    llm_total5 = 0.0
    if llm_ok5:
        t5l = time.time()
        first5l = None
        with httpx.stream(
            'POST', 'http://localhost:8000/v1/chat/completions',
            json={'model': 'qwen2.5-7b-instruct-fp8',
                  'messages': [{'role': 'user', 'content': stt_text5 or 'नमस्ते'}],
                  'max_tokens': 50, 'stream': True},
            timeout=60
        ) as r5l:
            for line5l in r5l.iter_lines():
                if line5l.startswith('data: '):
                    d5l = line5l[6:]
                    if d5l == '[DONE]': break
                    try:
                        ch5l = json.loads(d5l)
                        tok5l = ch5l['choices'][0]['delta'].get('content', '')
                        if tok5l:
                            if first5l is None: first5l = time.time()
                            llm_resp5 += tok5l
                    except Exception:
                        pass
        llm_ttft5 = (first5l - t5l) * 1000 if first5l else 0.0
        llm_total5 = (time.time() - t5l) * 1000
        print(f'  LLM: TTFT={llm_ttft5:.0f}ms  total={llm_total5:.0f}ms → {llm_resp5!r}')
    else:
        llm_resp5 = 'नमस्ते, मैं आपकी कैसे मदद कर सकती हूं?'
        print(f'  LLM: BLOCKED — fallback: {llm_resp5!r}')

    # TTS
    tts_chunks5 = []
    tts_ttfa5 = 0.0
    if tts_ok5:
        t5t = time.time()
        first5t = None
        with httpx.stream(
            'POST', 'http://localhost:8200/synthesize',
            json={'text': llm_resp5, 'speaker': 'Kavya'},
            timeout=120
        ) as r5t:
            for ch5t in r5t.iter_bytes(chunk_size=4096):
                if ch5t:
                    if first5t is None: first5t = time.time()
                    tts_chunks5.append(ch5t)
        tts_ttfa5 = (first5t - t5t) * 1000 if first5t else 0.0
        tts_total5 = (time.time() - t5t) * 1000
        print(f'  TTS: TTFA={tts_ttfa5:.0f}ms  total={tts_total5:.0f}ms  chunks={len(tts_chunks5)}')
    else:
        print('  TTS: BLOCKED')

    # AudioPacer
    sys.path.insert(0, VOICEOS)
    from deployment.gpu.audio.pacer import AudioPacer
    pacer5 = AudioPacer()
    for ch5p in tts_chunks5:
        pacer5.feed(ch5p)
    frames5 = list(pacer5.drain_all_frames())
    print(f'  AudioPacer: {len(frames5)} frames  ({len(frames5)*20}ms audio)')

    e2e_lat5 = (time.time() - t_pipe5) * 1000
    print(f'\n  E2E latency (audio in → first audio frame out): {e2e_lat5:.0f} ms')

    # Post-load VRAM
    print('\n[VRAM after full conversation]')
    if N_GPU >= 2:
        for gi5b in range(N_GPU):
            v5b = torch.cuda.memory_allocated(gi5b) / (1024 * 1024)
            print(f'  GPU {gi5b} VRAM: {v5b:.1f} MB')
    else:
        v5b = torch.cuda.memory_allocated(0) / (1024 * 1024)
        print(f'  GPU 0 VRAM: {v5b:.1f} MB')

    E2E_RESULTS = {
        'label': 'REAL — KAGGLE 2×T4',
        'mode': MODE5,
        'stt_ok': stt_ok5,
        'llm_ok': llm_ok5,
        'tts_ok': tts_ok5,
        'stt_latency_ms': stt_lat5,
        'llm_ttft_ms': llm_ttft5,
        'llm_total_ms': llm_total5,
        'tts_ttfa_ms': tts_ttfa5,
        'e2e_frames': len(frames5),
        'e2e_audio_ms': len(frames5) * 20,
        'e2e_latency_ms': e2e_lat5,
    }
    if stt_ok5 and tts_ok5:
        E2E_STATUS = 'PASS'
    elif stt_ok5 or tts_ok5:
        E2E_STATUS = 'PARTIAL'
    else:
        E2E_STATUS = 'BLOCKED'
    builtins.E2E_RESULTS = E2E_RESULTS
    builtins.E2E_STATUS = E2E_STATUS
    print(f'\nE2E_STATUS = {E2E_STATUS}')
    print(f'E2E_RESULTS = {E2E_RESULTS}')
    print('\n[CELL 5 COMPLETE — REAL — KAGGLE 2×T4]')

except SystemExit as _se5:
    print(str(_se5))
except Exception as _e5:
    import traceback, builtins
    E2E_STATUS = f'FAIL — {_e5}'
    E2E_RESULTS = {'error': str(_e5), 'label': 'REAL — KAGGLE 2×T4'}
    builtins.E2E_STATUS = E2E_STATUS
    builtins.E2E_RESULTS = E2E_RESULTS
    open('/kaggle/working/probe_cell5.txt', 'w').write(f'CELL 5 ERROR: {_e5}')
    print(f'CELL 5 ERROR: {_e5}')
    traceback.print_exc()
finally:
    for _p5, _n5 in [(_e2e_llm5, 'LLM'), (_e2e_stt5, 'STT'), (_e2e_tts5, 'TTS')]:
        if _p5 and _p5.poll() is None:
            _p5.terminate()
            try:
                _p5.wait(timeout=15)
            except Exception:
                _p5.kill()
            print(f'  {_n5} subprocess terminated')
    import torch as _t5; _t5.cuda.empty_cache()
    print('  CUDA cache emptied')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6: STEP 10 — AUDIO ARTIFACT
# REAL — KAGGLE 2×T4
# ═══════════════════════════════════════════════════════════════════
try:
    open('/kaggle/working/probe_cell6.txt', 'w').write('CELL 6 STARTED')

    import wave, struct, audioop, math, os, builtins

    print('=' * 70)
    print('REAL — KAGGLE 2×T4  |  CELL 6: STEP 10 — AUDIO ARTIFACT')
    print('=' * 70)

    wav_path6 = '/kaggle/working/veena_output_24khz.wav'
    ulaw_path6 = '/kaggle/working/veena_output_8khz.ulaw'

    if not os.path.exists(wav_path6):
        print(f'  WARNING: {wav_path6} not found — TTS cell was likely BLOCKED')
        print('  CELL 6: BLOCKED — no audio artifact to analyze')
    else:
        # ── Load WAV ─────────────────────────────────────────────────
        with wave.open(wav_path6, 'rb') as w6:
            sr6  = w6.getframerate()
            ch6  = w6.getnchannels()
            sw6  = w6.getsampwidth()
            nf6  = w6.getnframes()
            raw6 = w6.readframes(nf6)

        print(f'  WAV: {sr6}Hz  {ch6}ch  {sw6*8}bit  {nf6} frames')
        samples6 = struct.unpack(f'<{len(raw6)//2}h', raw6)
        n_samp6 = len(samples6)
        dur6_s = n_samp6 / sr6
        dur6_ms = dur6_s * 1000

        min6 = min(samples6) if samples6 else 0
        max6 = max(samples6) if samples6 else 0
        rms6 = math.sqrt(sum(s*s for s in samples6) / n_samp6) if n_samp6 > 0 else 0.0

        clip6 = sum(1 for s in samples6 if abs(s) > 32000)
        sil6  = sum(1 for s in samples6 if abs(s) < 100)
        clip_ratio6 = clip6 / n_samp6 if n_samp6 > 0 else 0.0
        sil_ratio6  = sil6  / n_samp6 if n_samp6 > 0 else 0.0

        print(f'\n[Audio quality metrics — REAL — KAGGLE 2×T4]')
        print(f'  Duration             : {dur6_ms:.0f} ms ({dur6_s:.3f} s)')
        print(f'  Sample count         : {n_samp6}')
        print(f'  Min sample           : {min6}')
        print(f'  Max sample           : {max6}')
        print(f'  RMS                  : {rms6:.1f}')
        print(f'  Is PCM16LE aligned   : YES (validated in Cell 4)')

        tts_results6 = getattr(builtins, 'TTS_RESULTS', {})
        float32_bug6 = tts_results6.get('float32_bug', 'N/A')
        print(f'  Has float32 bug      : {float32_bug6} (must be False)')

        print(f'  Clipping ratio       : {clip_ratio6:.4f} ({clip6} samples > |32000|)')
        print(f'  Silence ratio        : {sil_ratio6:.4f} ({sil6} samples < |100|)')

        pacer_results6 = getattr(builtins, 'PACER_RESULTS', {})
        underruns6 = pacer_results6.get('underruns', 'N/A')
        silence_frames6 = pacer_results6.get('silence_frames', 'N/A')
        total_frames6 = pacer_results6.get('total_frames', None)
        print(f'  Underruns            : {underruns6} (must be 0)')
        print(f'  Silence frames (μ-law): {silence_frames6}')

        # Inter-chunk gap analysis
        if isinstance(silence_frames6, int) and isinstance(total_frames6, int) and total_frames6 > 0:
            gap_ratio6 = silence_frames6 / total_frames6
            if silence_frames6 == 0:
                gap_analysis6 = 'NO silence gaps — words should sound CONTINUOUS'
            elif gap_ratio6 < 0.05:
                gap_analysis6 = f'MINIMAL silence gaps ({silence_frames6}/{total_frames6} = {gap_ratio6*100:.1f}%) — likely CONTINUOUS'
            else:
                gap_analysis6 = f'SIGNIFICANT silence gaps ({silence_frames6}/{total_frames6} = {gap_ratio6*100:.1f}%) — may sound BROKEN'
        else:
            gap_analysis6 = 'N/A (pacer data not available)'
        print(f'  Gap analysis         : {gap_analysis6}')

        # Intelligibility
        if (dur6_ms > 500
                and underruns6 == 0
                and sil_ratio6 < 0.3
                and rms6 > 500):
            intell6 = 'LIKELY INTELLIGIBLE'
        elif dur6_ms == 0:
            intell6 = 'NO AUDIO PRODUCED'
        else:
            intell6 = f'UNCERTAIN (dur={dur6_ms:.0f}ms rms={rms6:.0f} sil={sil_ratio6:.2f} underruns={underruns6})'
        print(f'  Estimated intelligibility: {intell6}')

        # ── Decode μ-law → 8kHz WAV ───────────────────────────────────
        if os.path.exists(ulaw_path6):
            print(f'\n[Decoding μ-law → 8kHz PCM WAV]')
            ulaw_bytes6 = open(ulaw_path6, 'rb').read()
            pcm_decoded6 = audioop.ulaw2lin(ulaw_bytes6, 2)  # → 16-bit PCM
            decoded_wav6 = '/kaggle/working/veena_output_8khz_decoded.wav'
            with wave.open(decoded_wav6, 'wb') as w6b:
                w6b.setnchannels(1)
                w6b.setsampwidth(2)
                w6b.setframerate(8000)
                w6b.writeframes(pcm_decoded6)
            print(f'  Saved: veena_output_8khz_decoded.wav ({len(pcm_decoded6)} bytes @ 8kHz)')
        else:
            print(f'  WARNING: {ulaw_path6} not found — skipping μ-law decode')

        print('\n' + '─' * 55)
        print('AUDIO ARTIFACT: Download veena_output_24khz.wav for listening (24kHz PCM16LE)')
        print('AUDIO ARTIFACT: Download veena_output_8khz_decoded.wav for listening (8kHz after telephony conversion)')
        print('─' * 55)
        print(f'\nGap analysis: {gap_analysis6}')
        if isinstance(silence_frames6, int) and silence_frames6 < 3:
            print('CONCLUSION: Words should sound CONTINUOUS — no AudioPacer underruns or silence gaps')
        else:
            print('CONCLUSION: Verify by listening to the WAV artifact')

    open('/kaggle/working/probe_cell6.txt', 'w').write('CELL 6 COMPLETE')
    print('\n[CELL 6 COMPLETE — REAL — KAGGLE 2×T4]')

except Exception as _e6:
    import traceback
    open('/kaggle/working/probe_cell6.txt', 'w').write(f'CELL 6 ERROR: {_e6}')
    print(f'CELL 6 ERROR: {_e6}')
    traceback.print_exc()


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7: STEPS 3+12 — MODEL COMPATIBILITY + FINAL REPORT
# REAL — KAGGLE 2×T4
# ═══════════════════════════════════════════════════════════════════
try:
    open('/kaggle/working/probe_cell7.txt', 'w').write('CELL 7 STARTED')

    import time, builtins

    print('=' * 70)
    print('REAL — KAGGLE 2×T4  |  CELL 7: STEPS 3+12 — FINAL REPORT')
    print('=' * 70)

    # Retrieve all results from previous cells
    STT_STATUS7   = getattr(builtins, 'STT_STATUS',   'NOT RUN')
    STT_RESULTS7  = getattr(builtins, 'STT_RESULTS',  {})
    LLM_STATUS7   = getattr(builtins, 'LLM_STATUS',   'NOT RUN')
    LLM_RESULTS7  = getattr(builtins, 'LLM_RESULTS',  {})
    TTS_STATUS7   = getattr(builtins, 'TTS_STATUS',   'NOT RUN')
    TTS_RESULTS7  = getattr(builtins, 'TTS_RESULTS',  {})
    PACER_STATUS7 = getattr(builtins, 'PACER_STATUS', 'NOT RUN')
    PACER_RESULTS7 = getattr(builtins, 'PACER_RESULTS', {})
    E2E_STATUS7   = getattr(builtins, 'E2E_STATUS',   'NOT RUN')
    E2E_RESULTS7  = getattr(builtins, 'E2E_RESULTS',  {})
    GPU_INFO7     = getattr(builtins, 'GPU_INFO', [])
    N_GPU7        = getattr(builtins, 'N_GPU', 0)

    now7 = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

    def _fmtf(val, unit='ms'):
        """Format a float value or return the raw value as string."""
        if isinstance(val, float):
            return f'{val:.0f} {unit}'
        return str(val)

    lines7 = []
    lines7.append('=' * 72)
    lines7.append('VOICEOS GPU VALIDATION REPORT')
    lines7.append('REAL — KAGGLE 2×T4')
    lines7.append(f'Generated: {now7}')
    lines7.append('=' * 72)
    lines7.append('')

    # ── GPU summary ──────────────────────────────────────────────────
    lines7.append('GPU ENVIRONMENT')
    lines7.append('-' * 40)
    lines7.append(f'N_GPU = {N_GPU7}')
    for g7 in GPU_INFO7:
        lines7.append(f'  GPU {g7["idx"]}: {g7["name"]}  VRAM={g7["vram_mb"]} MB  {g7["sm"]}  compute={g7["compute_capability"]}')
        lines7.append(f'    bf16_tensor_cores={g7["bf16_tensor_cores"]}  fp16={g7["fp16_tensor_cores"]}  fp8_hw={g7["fp8_hw"]}')
    if N_GPU7 >= 2:
        lines7.append('  GPU layout: GPU 0 → LLM  |  GPU 1 → STT+TTS')
    else:
        lines7.append('  GPU layout: GPU 0 → all services (single-GPU fallback)')
    lines7.append('')

    # ── STEP 3: Model Compatibility ──────────────────────────────────
    lines7.append('STEP 3 — MODEL COMPATIBILITY (T4 sm_75)')
    lines7.append('-' * 40)
    lines7.append('STT  | faster-whisper large-v3-turbo | compute_type=int8_float16')
    lines7.append('     | int8_float16 IS compatible with T4 sm_75 — VERIFIED')
    lines7.append('LLM  | RedHatAI/Qwen2.5-7B-Instruct-FP8-dynamic')
    lines7.append('     | sm_75 has no FP8 HW — vLLM dequantizes to BF16/FP16 at runtime')
    lines7.append('     | Same runtime behaviour as A6000 FP8 dequant path')
    lines7.append('TTS  | maya-research/Veena (3B BF16)')
    lines7.append('     | sm_75 has no BF16 tensor cores — FP32 emulation (slower, functional)')
    lines7.append('SNAC | hubertsiuzdak/snac_24khz')
    lines7.append('     | No hardware constraint — runs on any CUDA device')
    lines7.append('vLLM | --enforce-eager T4 fallback: CUDA graphs may fail on sm_75')
    lines7.append('')

    # ── STEP 4: STT ──────────────────────────────────────────────────
    lines7.append('STEP 4 — STT VALIDATION (REAL — KAGGLE 2×T4)')
    lines7.append('-' * 40)
    lines7.append(f'STATUS : {STT_STATUS7}')
    if STT_RESULTS7:
        lines7.append(f'Model  : {STT_RESULTS7.get("model", "N/A")}')
        lines7.append(f'GPU    : {STT_RESULTS7.get("gpu", "N/A")}')
        lines7.append(f'Load   : {_fmtf(STT_RESULTS7.get("load_ms"))}')
        lines7.append(f'Warmup : {_fmtf(STT_RESULTS7.get("warmup_ms"))}')
        trials7 = STT_RESULTS7.get('trial_latencies_ms', [])
        if trials7:
            lines7.append(f'Trials : {" / ".join(_fmtf(t7) for t7 in trials7)}')
        lines7.append(f'Avg lat: {_fmtf(STT_RESULTS7.get("avg_latency_ms"))}')
        lines7.append(f'VRAM Δ : {_fmtf(STT_RESULTS7.get("vram_delta_mb"), "MB")}')
    lines7.append('')

    # ── STEP 5: LLM ──────────────────────────────────────────────────
    lines7.append('STEP 5 — LLM VALIDATION (REAL — KAGGLE 2×T4)')
    lines7.append('-' * 40)
    lines7.append(f'STATUS : {LLM_STATUS7}')
    if LLM_RESULTS7:
        lines7.append(f'Model  : {LLM_RESULTS7.get("model", "N/A")}')
        lines7.append(f'vLLM   : {LLM_RESULTS7.get("vllm_version", "N/A")}')
        lines7.append(f'GPU    : {LLM_RESULTS7.get("gpu", "N/A")}')
        lines7.append(f'Startup: {_fmtf(LLM_RESULTS7.get("startup_ms"))}')
        lines7.append(f'TTFT   : {_fmtf(LLM_RESULTS7.get("ttft_ms"))}')
        lines7.append(f'Total  : {_fmtf(LLM_RESULTS7.get("total_latency_ms"))}')
        lines7.append(f'Prompt : {LLM_RESULTS7.get("prompt", "N/A")}')
        lines7.append(f'Response: {LLM_RESULTS7.get("response", "N/A")}')
        if LLM_RESULTS7.get('note'):
            lines7.append(f'Note   : {LLM_RESULTS7["note"]}')
    lines7.append('')

    # ── STEP 6: TTS ──────────────────────────────────────────────────
    lines7.append('STEP 6 — TTS VALIDATION (REAL — KAGGLE 2×T4)')
    lines7.append('-' * 40)
    lines7.append(f'STATUS : {TTS_STATUS7}')
    if TTS_RESULTS7:
        lines7.append(f'Model    : {TTS_RESULTS7.get("model", "N/A")}')
        lines7.append(f'GPU      : {TTS_RESULTS7.get("gpu", "N/A")}')
        lines7.append(f'Startup  : {_fmtf(TTS_RESULTS7.get("startup_ms"))}')
        lines7.append(f'TTFA     : {_fmtf(TTS_RESULTS7.get("ttfa_ms"))}')
        lines7.append(f'Chunks   : {TTS_RESULTS7.get("chunks", "N/A")}')
        lines7.append(f'Bytes    : {TTS_RESULTS7.get("total_bytes", "N/A")}')
        lines7.append(f'Duration : {_fmtf(TTS_RESULTS7.get("audio_duration_ms"))}')
        lines7.append(f'Float32 bug: {TTS_RESULTS7.get("float32_bug", "N/A")} (must be False)')
        lines7.append(f'Encoding : PCM16LE  chunk_bytes=4096  (validated)')
    lines7.append('')

    # ── STEP 7: Audio Pacer ──────────────────────────────────────────
    lines7.append('STEP 7 — AUDIO PACER VALIDATION (REAL — KAGGLE 2×T4)')
    lines7.append('-' * 40)
    lines7.append(f'STATUS : {PACER_STATUS7}')
    if PACER_RESULTS7:
        lines7.append(f'Frames   : {PACER_RESULTS7.get("total_frames", "N/A")} × 20ms = {PACER_RESULTS7.get("total_audio_ms", "N/A")} ms')
        lines7.append(f'Underruns: {PACER_RESULTS7.get("underruns", "N/A")} (must be 0 — 0 = no Na--mas--te gap)')
        lines7.append(f'Silence  : {PACER_RESULTS7.get("silence_frames", "N/A")} frames')
        lines7.append(f'Cancel   : {PACER_RESULTS7.get("cancel_test", "N/A")}')
        lines7.append('Contracts:')
        lines7.append('  4096 bytes / 2 = 2048 int16 samples = 85.33ms per TTS chunk')
        lines7.append('  160 bytes μ-law at 8kHz = 20ms per frame')
        lines7.append('  8192-byte chunks = float32 bug (must be absent)')
    lines7.append('')

    # ── STEPS 8+9: E2E ───────────────────────────────────────────────
    lines7.append('STEPS 8+9 — E2E PIPELINE (REAL — KAGGLE 2×T4)')
    lines7.append('-' * 40)
    lines7.append(f'STATUS : {E2E_STATUS7}')
    if E2E_RESULTS7:
        lines7.append(f'Mode     : {E2E_RESULTS7.get("mode", "N/A")}')
        lines7.append(f'STT ok   : {E2E_RESULTS7.get("stt_ok", "N/A")}')
        lines7.append(f'LLM ok   : {E2E_RESULTS7.get("llm_ok", "N/A")}')
        lines7.append(f'TTS ok   : {E2E_RESULTS7.get("tts_ok", "N/A")}')
        lines7.append(f'STT lat  : {_fmtf(E2E_RESULTS7.get("stt_latency_ms"))}')
        lines7.append(f'LLM TTFT : {_fmtf(E2E_RESULTS7.get("llm_ttft_ms"))}')
        lines7.append(f'TTS TTFA : {_fmtf(E2E_RESULTS7.get("tts_ttfa_ms"))}')
        lines7.append(f'E2E lat  : {_fmtf(E2E_RESULTS7.get("e2e_latency_ms"))}')
        lines7.append(f'Frames   : {E2E_RESULTS7.get("e2e_frames", "N/A")} ({E2E_RESULTS7.get("e2e_audio_ms", "N/A")} ms audio)')
    lines7.append('')

    # ── STEP 12 / Historical note ─────────────────────────────────────
    lines7.append('HISTORICAL CONTEXT')
    lines7.append('-' * 40)
    lines7.append('HISTORICAL — P100 (sm_60):')
    lines7.append('  Previous validation baseline was on P100 (sm_60).')
    lines7.append('  P100 results are NOT mixed into this report.')
    lines7.append('  This report contains ONLY REAL — KAGGLE 2×T4 measurements.')
    lines7.append('  P100 baseline is archived separately.')
    lines7.append('')

    # ── Final status summary ─────────────────────────────────────────
    lines7.append('=' * 72)
    lines7.append('FINAL STATUS SUMMARY — REAL — KAGGLE 2×T4')
    lines7.append('=' * 72)
    lines7.append(f'  STT         : {STT_STATUS7}')
    lines7.append(f'  LLM         : {LLM_STATUS7}')
    lines7.append(f'  TTS         : {TTS_STATUS7}')
    lines7.append(f'  AUDIO_PACER : {PACER_STATUS7}')
    lines7.append(f'  E2E         : {E2E_STATUS7}')
    lines7.append('')

    statuses7 = [STT_STATUS7, LLM_STATUS7, TTS_STATUS7, PACER_STATUS7, E2E_STATUS7]
    if any('FAIL' in str(s) for s in statuses7):
        overall7 = 'FAIL'
    elif any('BLOCKED' in str(s) for s in statuses7):
        n_blocked7 = sum(1 for s in statuses7 if 'BLOCKED' in str(s))
        overall7 = f'BLOCKED ({n_blocked7} component(s))'
    elif all(str(s).startswith('PASS') for s in statuses7):
        overall7 = 'PASS'
    else:
        overall7 = 'PARTIAL'

    lines7.append(f'FINAL OVERALL: {overall7}')
    lines7.append('')
    lines7.append('GPU Assignment:')
    lines7.append('  GPU 0 → LLM (vLLM)')
    lines7.append('  GPU 1 → STT + TTS  (if N_GPU >= 2)')
    lines7.append('  Single-GPU fallback: all on GPU 0  (if N_GPU < 2)')
    lines7.append('')
    lines7.append('T4 Notes:')
    lines7.append('  sm_75: No BF16 tensor cores (FP32 emulation for Veena BF16 — slower but functional)')
    lines7.append('  sm_75: No FP8 hardware (Qwen FP8 dequantized at runtime — same path as A6000)')
    lines7.append('  sm_75: vLLM --enforce-eager fallback applied (CUDA graphs may fail on sm_75)')
    lines7.append('  int8_float16 STT: fully compatible with T4 sm_75')
    lines7.append('')
    lines7.append(f'Report: {now7}')
    lines7.append('REAL — KAGGLE 2×T4')

    report7 = '\n'.join(lines7)
    print(report7)

    with open('/kaggle/working/validation_report.txt', 'w') as f7:
        f7.write(report7 + '\n')
    print('\nFull report saved to /kaggle/working/validation_report.txt')

    open('/kaggle/working/probe_cell7.txt', 'w').write(f'CELL 7 COMPLETE — {overall7}')
    print('\n[CELL 7 COMPLETE — REAL — KAGGLE 2×T4]')

except Exception as _e7:
    import traceback
    open('/kaggle/working/probe_cell7.txt', 'w').write(f'CELL 7 ERROR: {_e7}')
    print(f'CELL 7 ERROR: {_e7}')
    traceback.print_exc()
    raise
